# Archived decoding comparisons

Compare signal representations, decoder choices, time windows, and CEBRA settings using the checked-in historical scores. This is separate from the poster’s fixed 500 ms comparison. No new fitting or test scoring takes place.

Balanced accuracy is computed within each session and averaged with equal session weight. Validation includes all five sessions. Historical strict test includes the four sessions with all five target classes; the pooling comparison uses those same four sessions on both splits. The test was previously viewed. No test data are loaded here.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt

here = Path.cwd().resolve()
root = next(p for p in (here, *here.parents) if (p / 'src').is_dir() and (p / 'poster').is_dir())
sys.path.insert(0, str(root / 'src'))
import pandas as pd
from IPython.display import display

scores = pd.read_csv(root / 'presentation/evidence/model_scores.csv', keep_default_na=False)
names = {'raw': 'Raw LFP', 'amp': 'Wavelet amplitude', 'phase': 'Wavelet phase',
         'combined': 'Amplitude + phase', 'cebra_raw': 'Raw LFP + CEBRA',
         'cebra_combined': 'Wavelets + CEBRA', 'raw_binned': 'Raw LFP, binned'}

def score(split, target, duration, rep, *, start=0, decoder='l1', pool=None, steps=None, arch='offset10-model', seed=42):
    latent = rep.startswith('cebra_')
    pool = pool or ('mean' if latent else 'default')
    steps = steps if steps is not None else (1000 if latent else 0)
    arch = arch if latent else ''
    row = scores.query('split == @split and target == @target and start == @start and duration == @duration and '
                       'rep == @rep and pool == @pool and arch == @arch and steps == @steps and '
                       'seed == @seed and decoder == @decoder')
    return 100 * row.mean_ba.item()

def bars(title, rows):
    labels, values = zip(*rows, strict=True)
    fig, ax = plt.subplots(figsize=(9, max(3, .43*len(rows))), layout='constrained')
    marks = ax.barh(labels, values, color='#128170')
    ax.bar_label(marks, fmt='%.1f%%', padding=3)
    ax.axvline(20, color='gray', linewidth=.8, linestyle=':')
    ax.set(title=title, xlabel='Mean balanced accuracy (%)', xlim=(0, max(60, max(values)+8)))
    ax.invert_yaxis()

## Decoder baselines and onset-aligned historical test

Cosine 3-nearest neighbors (3-NN) is a Stage 1 baseline; the main comparisons use L1-regularized logistic regression. The position CEBRA finalist pooled four 50 ms bins, while its baseline used a whole-window mean. These are different configurations.

In [ ]:
compare = scores.query('split == "validation" and stage == 1').copy()
compare['balanced_accuracy_pct'] = 100 * compare.mean_ba
baseline = compare.pivot(index=['target', 'duration', 'rep'], columns='decoder', values='balanced_accuracy_pct')
display(baseline.rename(index=names, level='rep').round(1))
test = 'historical_test'
bars('Position · selected representations, historical test',
     [(f'{names[rep]} · 0–{d} ms', score(test, 'position', d, rep))
      for d in (200, 500) for rep in ('combined', 'cebra_raw')])

fig, ax = plt.subplots(figsize=(9, 3), layout='constrained')
for j, (direct, latent) in enumerate((('raw', 'cebra_raw'), ('combined', 'cebra_combined'))):
    a, b = [score(test, 'position', 500, rep) for rep in (direct, latent)]
    ax.plot([a, b], [j, j], color='0.5')
    ax.scatter(a, j, facecolors='white', edgecolors='0.4', label='Direct' if j == 0 else None, zorder=3)
    ax.scatter(b, j, color='#128170', label='With CEBRA' if j == 0 else None, zorder=3)
    ax.text(max(a, b)+1, j, f'{b-a:+.1f} points', va='center')
ax.set(title='Position · with and without CEBRA, 500 ms historical test', xlim=(15, 64), ylim=(-.5, 1.5),
       yticks=(0, 1), yticklabels=(names['raw'], names['combined']), xlabel='Mean balanced accuracy (%)')
ax.legend();

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True, layout='constrained')
reps = list(names)[:6]
for ax, d in zip(axes, (200, 500), strict=True):
    values = [score(test, 'odor', d, rep) for rep in reps]
    lines = ax.barh([names[rep] for rep in reps], values, color=['#128170' if r == 'amp' else '#9cb7b2' for r in reps])
    ax.bar_label(lines, fmt='%.1f%%', padding=3)
    ax.axvline(20, color='gray', linestyle=':')
    ax.set(title=f'Odor · 0–{d} ms historical test', xlabel='Mean balanced accuracy (%)', xlim=(0, 60))
axes[0].invert_yaxis()

bars('Historical finalist test scores', [
    ('Position · amplitude 0–500 ms', score(test, 'position', 500, 'amp')),
    ('Position · wavelets + CEBRA 0–200 ms, 50 ms bins', score(test, 'position', 200, 'cebra_combined', pool='bin50')),
    ('Odor · amplitude 0–500 ms', score(test, 'odor', 500, 'amp')),
    ('Odor · wavelets + CEBRA 0–500 ms, mean', score(test, 'odor', 500, 'cebra_combined'))]);

## Direct-feature validation windows

The run compares five 100 ms windows, four 200 ms windows, and one 500 ms window. The highest validation score for each target used 500 ms amplitude; this does not establish the best test configuration.

In [ ]:
windows = [(start, d) for d in (100, 200, 500) for start in range(0, 501-d, 100)]
direct = list(names)[:4]
val = 'validation'
fig, axes = plt.subplots(2, 1, figsize=(12, 6), layout='constrained')
for ax, target in zip(axes, ('odor', 'position'), strict=True):
    values = np.array([[score(val, target, d, rep, start=start) for start, d in windows] for rep in direct])
    im = ax.imshow(values, cmap='BuGn', vmin=20, vmax=55, aspect='auto')
    for i, j in np.ndindex(values.shape):
        top = values[i, j] == values.max()
        ax.text(j, i, f'{values[i,j]:.1f}', ha='center', va='center', weight='bold' if top else 'normal')
    ax.set(title=f'{target.capitalize()} · five-session mean balanced accuracy (%)', xticks=range(len(windows)),
           xticklabels=[f'{start}–{start+d}' for start, d in windows], yticks=range(len(direct)),
           yticklabels=[names[rep] for rep in direct])
axes[-1].set_xlabel('Time after odor onset (ms)')
fig.colorbar(im, ax=axes, label='Mean balanced accuracy (%)');

## CEBRA settings and added windows

Pooling, update count, seed, and architecture are separate comparisons. The four-rat pairing is only for the pooling panel. The five-rat architecture and update comparisons use validation, as does the CEBRA sweep over the eight added windows.

In [ ]:
def latent(target, duration, split=val, **kwargs):
    return score(split, target, duration, 'cebra_combined', **kwargs)

bars('Validation · encoder architecture', [
    (f'{target.capitalize()} · {arch}', latent(target, d, pool=pool, arch=arch+'-model'))
    for target, d, pool in [('position', 200, 'bin50'), ('odor', 500, 'mean')]
    for arch in ('offset10', 'offset5')])

bars('Position · different temporal pooling, same four rats', [
    (f'{label} · {name}', latent('position', 200, split, pool=pool))
    for label, split in [('Validation', 'paired_validation'), ('Historical test', test)]
    for name, pool in [('window mean', 'mean'), ('50 ms bins', 'bin50')]])

bars('Position · CEBRA training stability, validation', [
    (f'{steps:,} updates', latent('position', 200, pool='bin50', steps=steps)) for steps in (1000, 5000)
] + [(f'Seed {seed}', latent('position', 200, pool='bin50', seed=seed)) for seed in (42, 43, 44)])

added = scores.query('split == "validation" and stage == 4 and arch == "offset10-model"').copy()
added['end'] = added.start + added.duration
added['mean_ba_pct'] = added.mean_ba * 100
display(added.sort_values(['target', 'mean_ba'], ascending=[True, False])
        [['target', 'rep', 'start', 'end', 'mean_ba_pct']].round(1));

The [experiment guide](../presentation/README.md) explains how to regenerate the aggregate from authorized scores and documents the four-stage ablation and later encoder–decoder designs.